### Imports

In [7]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os
import json
from typing import Dict, List, Tuple, Optional

In [8]:
import sys
sys.path.append("..")

from scripts.eda import basic_summary, plot_series, plot_distribution, plot_rolling_statistics, plot_acf_series, plot_pacf_series, plot_hourly_pattern, plot_dayofweek_pattern, plot_boxplot_by_hour, plot_boxplot_by_dayofweek

### Data Analysis Pipeline

The identified files will be loaded below, and each will undergo a basic time series analysis, along with some relevant plotting. This file will conclude with a brief discussion of observations and potential modeling constraints.

Files to load:
- **CPU_UTIL_1** - ec2_cpu_utilization_5f5533.csv
- **CPU_UTIL_2** - ec2_cpu_utilization_53ea38.csv
- **DISK_WRITE_1** - ec2_disk_write_bytes_c0d644.csv
- **DISK_WRITE_2** - ec2_disk_write_bytes_1ef3de.csv
- **NETWORK_IN_1** - ec2_network_in_5abac7.csv
- **NETWORK_IN_2** - ec2_network_in_257a54.csv
- **REQUEST_COUNT** - elb_request_count_8c0756.csv

Note that in all cases, the data is structured with time-stamps and a singular value, meaning that this is not a **multi-variate time-series analysis**.

In [9]:
files_map = {
    'CPU_UTIL_1': 'ec2_cpu_utilization_5f5533.csv',
    'CPU_UTIL_2': 'ec2_cpu_utilization_53ea38.csv',
    'DISK_WRITE_': 'ec2_disk_write_bytes_c0d644.csv',
    'DISK_WRITE_2': 'ec2_disk_write_bytes_1ef3de.csv',
    'NETWORK_IN_1': 'ec2_network_in_5abac7.csv',
    'NETWORK_IN_2': 'ec2_network_in_257a54.csv',
    'REQUEST_COUNT': 'elb_request_count_8c0756.csv'
}

DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), 'data')

In [10]:
with open(os.path.join(DATA_DIR, 'combined_windows.json')) as file:
    windows_data = json.load(file)

anomaly_windows_map: Dict = {
    key: [
        (pd.Timestamp(start), pd.Timestamp(end))
        for start, end in windows_data.get(f'realAWSCloudwatch/{file}', [])
    ]
    for key, file in files_map.items()
}

dfs_map: Dict[str, Tuple[pd.DataFrame, List[Tuple[pd.Timestamp, pd.Timestamp]]]] = {}

for key, file in files_map.items():
    df = pd.read_csv(os.path.join(DATA_DIR, files_map[key]))
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = df.sort_values('timestamp').reset_index(drop=True)
    dfs_map[key] = (df, anomaly_windows_map[key])

In [ ]:
# Invokes visualization methods defined earlier for a comprehensive understanding of the dataset.
def summarize(dataset_name: str, df: pd.DataFrame, anomaly_windows: List[Tuple[pd.Timestamp, pd.Timestamp]]):
    basic_summary(df, True)
    plot_series(df, anomaly_windows, f'Time Series Plot (With Anomalies) for {key}')
    plot_distribution(df)
    plot_rolling_statistics(df, 24, f'Rolling Statistics (Window = 24 hours) Plot for {key}')
    plot_rolling_statistics(df, 48, f'Rolling Statistics (Window = 48 hours) Plot for {key}')
    plot_acf_series(df, 100, f'ACF Plot for {key}')
    plot_pacf_series(df, 50, f'PACF Plot for {key}')
    plot_hourly_pattern(df, f'Average Hourly Value for {key}')
    plot_dayofweek_pattern(df, f'Average Value by Day for {key}')
    plot_boxplot_by_hour(df, f'Hourly Box-Plot for {key}')
    plot_boxplot_by_dayofweek(df, f'Day-of-Week Box-Plot for {key}')

In [ ]:
for key, pair in dfs_map.items():
    summarize(key, pair[0], pair[1])